<!--
Licensed to the Apache Software Foundation (ASF) under one or more
contributor license agreements.  See the NOTICE file distributed with
this work for additional information regarding copyright ownership.
The ASF licenses this file to You under the Apache License, Version 2.0
(the "License"); you may not use this file except in compliance with
the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 08 · E8 — Copilot path vs deterministic endpoints

The product onboards/enriches through the **Copilot stream**, not the deterministic
endpoints notebooks 03/04 use. This drives the Copilot and compares the resulting
MDL (and graded score) to the deterministic path.

The glossary is handed to the Copilot as a structured **attachment**
(`v2.text_attachment`), which the server renders into the prompt under a
`### bi_glossary.md` header — the same `MessageAttachment` channel the UI uses — so
the document is separated from the instruction (EVAL_V2_SPEC.md E8).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import eval_common as ec
import eval_v2 as v2
import seagate_scoring as scoring

FIXTURE = 'seagate_multi'
fdir = v2.fixture_dir(FIXTURE)
manifest = v2.load_table_manifest(FIXTURE)

# Multi-schema project: primary schema seagate_ops + the seagate_core scope.
# seagate_ref is deliberately EXCLUDED (pulling it in is an E9 failure).
config = ec.EvalConfig.from_env(
    schema_name='seagate_ops',
    results_dir=fdir.parent.parent / 'evaluation' / 'results' / 'seagate_multi',
)
client = v2.AgentClientV2(config, schema_names=['seagate_core', 'seagate_ops'])
me = client.login()
print('Authenticated as:', me.get('username') or me)

# Preconditions: Postgres required (R4); memory loop must be off (R1, warn-only).
pre = client.assert_eval_preconditions(require_postgres=True)
print('DB backend:', pre['backend'])
for w in pre['warnings']:
    print('WARNING:', w)

questions = ec.parse_test_queries(fdir / 'test_queries.md')
print('Loaded', len(questions), 'graded questions (Q1-Q18)')
glossary = (fdir / 'bi_glossary.md').read_text(encoding='utf-8')

Authenticated as: admin


DB backend: postgresql
Loaded 18 graded questions (Q1-Q18)


In [2]:
def score_sweep(results):
    """Score a result set with the exact ground-truth scorer (incl. Q16-Q18)."""
    verdicts = {r['id']: scoring.score_result(
        r['id'], r['result_rows'], r['answer_summary']) for r in results}
    correct = sum(1 for v in verdicts.values() if v in scoring.CORRECT_VERDICTS)
    return correct, verdicts

### Resolve + onboard, then drive one Copilot enrichment turn

In [3]:
print('archived', client.clean_baseline(), 'project(s)')
project = client.resolve_project(); pid = project['id']
client.onboard(pid)

# One Copilot enrichment turn on the deterministically-onboarded project, via the
# robust build helper (turn -> apply -> activate; captures a 422 instead of raising).
# The glossary is handed over as a structured attachment (v2.text_attachment).
build = client.copilot_build(pid, v2.COPILOT_ENRICH_MESSAGE, glossary=glossary, max_steps=16)
print('changeset items=%s applied=%s activated=%s err=%s' % (
    build['items'], build['applied'], build['activated'], build['activate_error']))
print('progress steps:', sum(1 for e in build['events'] if e.get('type') == 'progress'))

archived 1 project(s)


changeset items: 5
progress steps: 21


### Apply the changeset (drafts) and activate, then grade

In [4]:
if build['activated']:
    cov = client.wait_for_coverage(pid)
    correct, verdicts = score_sweep(
        ec.run_experiment(client, 'wren_bi', questions, save=False))
    print(f'copilot wren_bi: coverage={cov} correct={correct}/18')
    print('selection metrics:', client.selection_metrics(pid, manifest))
elif build['applied']:
    print('Applied but NOT activated:', build['activate_error'])
    print('(needs an agent image exposing mdl-files/bulk-status — see RESULTS_v2 E8.)')
else:
    print('No changeset to apply:', build.get('error'))

AgentError: PATCH mdl-files -> 422: {"detail":"Cannot activate an MDL file with validation errors: MDL must contain at least one model, view, metric, or cube."}

**Compare** these numbers to notebook 06/04's deterministic `wren_bi`. A large
gap means the legacy eval does not describe what users actually get from the product
Copilot. The `selection_metrics` here double as an E9 signal on the Copilot path.